# Summarising results

This notebook summarises results into a Pandas dataframe which is then
reformatted into a table suitable for publishing.


In [1]:
from datetime import datetime, timezone
from typing import Any, Literal, TypeAlias

import pandas as pd

from qaoa_parameter_setting.utils.database import ResultsDatabase
from qaoa_parameter_setting.utils.database.summary_table_formatter import (
    convert_evaluation_to_multicolumn_latex,
    formatted_styler_for,
)


In [ ]:
# Make sure to change this when regenerating all tables.
PROBLEM_CLASS: Literal["MC", "MIS"] = "MC"

In [3]:
# TABLE_JSON: str | None = f"summary_tables_{PROBLEM_CLASS}.json"
TABLE_JSON: str | None = None
table = ResultsDatabase(TABLE_JSON, problem_class=PROBLEM_CLASS)

## Setup table with training data and min-max cuts.


In [4]:
if TABLE_JSON is None:

    def ignore_pt_aaa_results(filename: str, results: dict[str, Any]) -> bool:
        """Ignore Parameter Transfer results for the tables."""
        if "PT_" in filename:
            return True
        return False

    # Add training data. Only the "best" results are kept, per graph, trainer config
    # file, and depth.
    for path in [
        "../data/training/random_regular",
        "../data/training/heavy_hex",
        "../data/training/line_to_full",
        "../data/training/erdos_renyi",
    ]:
        table.add_data(path, ignore_file_function=ignore_pt_aaa_results)

In [ ]:
if TABLE_JSON is None and table.problem_class == "MC":
    # Add min- and max-cut data. If some graph instances do not have a minmax_cuts
    # entry, an error will be thrown later.
    table.add_minmax_cut_data("../data/minmax_cuts/random_regular")
    table.add_minmax_cut_data("../data/minmax_cuts/heavy_hex")
    table.add_minmax_cut_data("../data/minmax_cuts/line_to_full")
    table.add_minmax_cut_data("../data/minmax_cuts/erdos_renyi")

/tmp/ipykernel_253759/277003374.py:4: UserWarning: Loading min-max cut data but problem_class is 'MIS'. Min-max cut data is only applicable to MaxCut problems.
  table.add_minmax_cut_data("../data/minmax_cuts/random_regular")
/tmp/ipykernel_253759/277003374.py:5: UserWarning: Loading min-max cut data but problem_class is 'MIS'. Min-max cut data is only applicable to MaxCut problems.
  table.add_minmax_cut_data("../data/minmax_cuts/heavy_hex")
/tmp/ipykernel_253759/277003374.py:6: UserWarning: Loading min-max cut data but problem_class is 'MIS'. Min-max cut data is only applicable to MaxCut problems.
  table.add_minmax_cut_data("../data/minmax_cuts/line_to_full")
/tmp/ipykernel_253759/277003374.py:7: UserWarning: Loading min-max cut data but problem_class is 'MIS'. Min-max cut data is only applicable to MaxCut problems.
  table.add_minmax_cut_data("../data/minmax_cuts/erdos_renyi")


## Identify missing min- and max-cuts data


In [21]:
if table.problem_class == "MC":
    missing_minmax_cuts = table.missing_minmax_cuts()
    if len(missing_minmax_cuts) == 0:
        print("All minmax_cuts data accounted for.")
    else:
        print(
            "The following graphs are missing min- and max-cuts data. "
            + "Generate them with compute_min_max_for_graph.py"
        )
        for _graph in missing_minmax_cuts:
            print("- {}".format(_graph))
else:
    print(f"Min-/Max-cut data not necessary for {table.problem_class} data.")

Min-/Max-cut data not necessary for MIS data.


## List of methods per evaluation type

In [22]:
table.print_methods_by_evaluation()

      MPS (Aer)        |     MPS (Quimb)     |         PP         |         SV        
--------------------------------------------------------------------------------------
FA_MPSAer_no_opt.json  | FA_MPS_no_opt.json  | FA_PP_no_opt.json  | FA_SV_no_opt.json 
FA_MPSAer_opt.json     | FA_MPS_opt.json     | FA_PP_opt.json     | FA_SV_opt.json    
F_MPSAer.json          | F_MPS.json          | F_PP.json          | F_SV.json         
I_MPSAer.json          | I_MPS.json          | I_PP.json          | I_SV.json         
TQA_MPSAer_no_opt.json | TQA_MPS_no_opt.json | TQA_PP_no_opt.json | LR_SV_opt.json    
TQA_MPSAer_opt.json    | TQA_MPS_opt.json    | TQA_PP_opt.json    | TQA_SV_no_opt.json
                       |                     |                    | TQA_SV_opt.json   
                       |                     |                    | TS_SV.json        


## Save the database if we created a new database

In [8]:
if TABLE_JSON is None:
    TABLE_JSON = f"summary_tables_{table.problem_class}.json"
    table.save(TABLE_JSON, overwrite=True)

## Get raw data table


In [9]:
# Only select results with the best energy per config and instance:
table = table.only_best_parameters("config")

# This is the _raw_ table with all results
df: pd.DataFrame = table.to_dataframe()
df

,instance,num_nodes,graph_type,trainer_config,method,depth,energy,trainer,evaluation,evaluation_label,method_label,with_aer,source_file,train_duration,metadata,result_index,run_datetime,result_key_index,approximation_ratio
0,002_40nodes_random3regular.json,40,random_regular,F_MPS.json,F.json,1,37.092711,ScipyTrainer,MPS,MPS (Quimb),Fourier*,False,../data/training/random_regular/20251206_05112...,1.357959,"{'iteration': '1', 'version': 21}",0,2025-12-06 05:11:29,1.0,NaN
1,002_40nodes_random3regular.json,40,random_regular,F_MPS.json,F.json,2,66.570735,ScipyTrainer,MPS,MPS (Quimb),Fourier*,False,../data/training/random_regular/20251206_05112...,366.692246,"{'iteration': '2', 'version': 21}",0,2025-12-06 05:11:29,2.2,NaN
2,002_40nodes_random3regular.json,40,random_regular,F_MPS.json,F.json,3,69.476384,ScipyTrainer,MPS,MPS (Quimb),Fourier*,False,../data/training/random_regular/20251206_05112...,632.105480,"{'iteration': '3', 'version': 21}",0,2025-12-06 05:11:29,3.2,NaN
3,002_40nodes_random3regular.json,40,random_regular,F_MPS.json,F.json,4,69.799565,ScipyTrainer,MPS,MPS (Quimb),Fourier*,False,../data/training/random_regular/20251125_10345...,40.463099,"{'iteration': '4', 'version': 13}",0,2025-11-25 10:34:57,4.2,NaN
4,002_40nodes_random3regular.json,40,random_regular,F_MPS.json,F.json,5,69.825606,ScipyTrainer,MPS,MPS (Quimb),Fourier*,False,../data/training/random_regular/20251206_05112...,1100.812536,"{'iteration': '5', 'version': 21}",0,2025-12-06 05:11:29,5.2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13318,009_20nodes_erdosrenyi40percent.json,20,erdos_renyi,TQA_SV_opt.json,TQA_opt.json,6,102.505732,ScipyTrainer,SV,SV,TQA*,False,../data/training/erdos_renyi/20260416_001031_0...,232.622555,"{'iteration': '1', 'version': 33}",0,2026-04-16 00:10:31,1.0,NaN
13319,009_20nodes_erdosrenyi40percent.json,20,erdos_renyi,TQA_SV_no_opt.json,TQA_no_opt.json,6,60.173781,TQATrainer,SV,SV,TQA,False,../data/training/erdos_renyi/20260416_001031_0...,45.725976,"{'iteration': '0', 'version': 33}",0,2026-04-16 00:10:31,0.0,NaN
13320,009_20nodes_erdosrenyi50percent.json,20,erdos_renyi,LR_SV_opt.json,LR_opt.json,6,139.537627,TQATrainer,SV,SV,Linear Ramp,False,../data/training/erdos_renyi/20260416_000723_0...,52.887043,"{'iteration': '0', 'version': 33}",0,2026-04-16 00:07:23,0.0,NaN
13321,009_20nodes_erdosrenyi50percent.json,20,erdos_renyi,TQA_SV_opt.json,TQA_opt.json,6,76.035210,ScipyTrainer,SV,SV,TQA*,False,../data/training/erdos_renyi/20260416_001152_0...,268.015460,"{'iteration': '1', 'version': 33}",0,2026-04-16 00:11:52,1.0,NaN


## Get formatted and pivoted table and styler

With Pandas, dataframes are formatted with Stylers.
`qaoa_parameter_setting.utils.database.summary_table_formatter.formatted_styler_for` automatically
stylises the Styler and returns the pivoted dataframe and Styler.


### Example summary table with `"text"` formatting

The following cells format the summary results into a single table suitable for
Jupyter notebooks. Later cells will format the table for LaTeX and save them to
a file.

As we can generate tables for both MAXCUT and MIS, we use `performance_metric`
to store the field name used to quantify performance.


In [10]:
performance_metric: Literal["approximation_ratio", "energy"]
if table.problem_class == "MC":
    performance_metric = "approximation_ratio"
elif table.problem_class == "MIS":
    performance_metric = "energy"
else:
    raise NotImplementedError(
        "Performance summary tables not implemented for {!r} problem class.".format(
            table.problem_class
        )
    )

#### MAXCUT Approximation Ratio or MIS Normalized Energy


In [11]:
pivot, styler, _ = formatted_styler_for(
    table,
    depths=10,
    agg_values=performance_metric,
    with_fancy_values=False,
    cmap="Greens",
    precision=10,
    missing_data_str="-",
    target_format="text",
    show_empty_rows=True,
    exclude_methods=["RTS"],
)
styler

In [12]:
pivot, styler, _ = formatted_styler_for(
    table,
    depths=2,
    agg_values=performance_metric,
    with_fancy_values=True,
    cmap={"MPS (Aer)": "BuPu", "MPS (Quimb)": "YlOrBr", "PP": "YlGn", "SV": "PuBu"},
    precision=2,
    missing_data_str="-",
    target_format="text",
    show_empty_rows=True,
    exclude_methods=["RTS", "TS"],
)
styler

#### Number of instances and nodes


In [13]:
pivot, styler, _ = formatted_styler_for(
    table,
    depths=10,
    num_nodes=None,
    agg_values="num_instances",
    with_fancy_values=True,
    cmap="Greens",
    precision=0,
    missing_data_str="-",
    target_format="text",
    show_empty_rows=True,
    exclude_methods=["TS", "RTS"],
)
styler

## Combined table


In [14]:
table_cmap = {
    (k1, k2): v if k2 == performance_metric else "RdPu"
    for k1, v in {
        "MPS (Aer)": "BuPu",
        "MPS (Quimb)": "YlOrBr",
        "PP": "YlGn",
        "SV": "PuBu",
    }.items()
    for k2 in [performance_metric, "num_instances"]
}
pivot, styler, cmap_ranges = formatted_styler_for(
    table,
    depths=10,
    agg_values=[performance_metric, "num_instances"],
    with_fancy_values=True,
    cmap=table_cmap,
    precision={performance_metric: 2, "num_instances": 0},
    missing_data_str="-",
    target_format="text",
    show_empty_rows=False,
    exclude_methods=["RTS"],
)
styler

In [15]:
table_cmap = {
    (k1, k2): v if k2 == performance_metric else "RdPu"
    for k1, v in {
        "MPS (Aer)": "BuPu",
        "MPS (Quimb)": "YlOrBr",
        "PP": "YlGn",
        "SV": "PuBu",
    }.items()
    for k2 in [performance_metric, "num_instances"]
}
pivot, styler, cmap_ranges = formatted_styler_for(
    table,
    depths=6,
    num_nodes={
        "MPS (Aer)": [70, 100],
        "MPS (Quimb)": [70, 100],
        "PP": [70, 100],
        "SV": 20,
    },
    agg_values=[performance_metric, "num_instances"],
    with_fancy_values=True,
    cmap=table_cmap,
    precision={performance_metric: 2, "num_instances": 0},
    missing_data_str="-",
    target_format="text",
    exclude_methods=["RTS"],
)
styler

## Create and save tables for LaTeX

Here we format tables into LaTeX files and save them into separate `.tex` files
suitable for inclusion in a paper. Files are saved with the current date and
time for tracking changes. These can be compiled into a preview of all tables
with `pdflatex summary_tables.tex`


In [16]:
AggValue: TypeAlias = Literal["num_instances", "approximation_ratio", "energy"]
values_to_plot: list[
    tuple[AggValue | Literal["both"], AggValue | list[AggValue], str, str]
] = [
    (
        "num_instances",
        "num_instances",
        "Number of Instances (num. nodes in brackets)",
        " Cells are colored based on the number of instances for the given configuration, with darker colors indicating more instances.",
    )
]
# Define values and captions for tables
if table.problem_class == "MC":
    values_to_plot.extend(
        [
            (
                "approximation_ratio",
                "approximation_ratio",
                r"Avg. Approx. Ratio $\pm$ standard deviation in percentage for MAXCUT",
                " Evaluation methods are shown in different colors, with darker colors indicating better approximation ratios.",
            ),
            (
                "both",
                ["approximation_ratio", "num_instances"],
                "Avg. Approx. Ratio and Number of Instances for Various Evaluation Methods for MAXCUT",
                " Evaluation methods are shown in different colors for approximation ratios."
                + " Darker colors indicating better approximation ratios or more instances."
                + " Approximation ratios are in percentage, with $\\pm$ their standard deviation."
                + " The range of graph sizes are shown in brackets, next to the number of instances.",
            ),
        ]
    )
elif table.problem_class == "MIS":
    values_to_plot.extend(
        [
            (
                "energy",
                "energy",
                r"Avg. Energy $\pm$ standard deviation for MIS",
                " Evaluation methods are shown in different colors, with darker colors indicating better energies."
                + " The minimum and maximum energies defining the range of colors are shared between MPS and PP."
                + " Note that energies include invalid solutions, which are penalized by the Hamiltonian.",
            ),
            (
                "both",
                ["energy", "num_instances"],
                "Avg. Energy and Number of Instances for Various Evaluation Methods for MIS",
                " Evaluation methods are shown in different colors for energies."
                + " Darker colors indicating better energies or more instances."
                + " The minimum and maximum energies defining the range of colors are shared between MPS and PP."
                + " Average energies are shown, with $\\pm$ their standard deviation."
                + " The range of graph sizes are shown in brackets, next to the number of instances.",
            ),
        ]
    )

In [17]:
## Create the CMAPs for each table.
TABLE_CMAPS_PER_EVALUATION = {
    "MPS (Aer)": "BuPu",
    "MPS (Quimb)": "YlOrBr",
    "PP": "YlGn",
    "SV": "PuBu",
}
table_cmap_num_instances = "Greens"
# The combined tables use different ranges for SV. MAXCUT has the same range for all evaluation
# methods, whereas SV has its own range for MIS data.
if table.problem_class == "MC":
    table_cmap_performance = TABLE_CMAPS_PER_EVALUATION
    table_cmap_combined = [
        {
            (str(k1), performance_metric): v
            for k1, v in TABLE_CMAPS_PER_EVALUATION.items()
        },
        {
            (str(k1), "num_instances"): "RdPu"
            for k1 in TABLE_CMAPS_PER_EVALUATION.keys()
        },
    ]
else:
    table_cmap_performance = [
        {"MPS (Aer)": "BuPu", "MPS (Quimb)": "YlOrBr", "PP": "YlGn"},
        {"SV": "PuBu"},
    ]
    table_cmap_combined = [
        {
            (str(k1), performance_metric): v
            for k1, v in TABLE_CMAPS_PER_EVALUATION.items()
            if k1 != "SV"
        },
        {
            (str(k1), performance_metric): v
            for k1, v in TABLE_CMAPS_PER_EVALUATION.items()
            if k1 == "SV"
        },
        {
            (str(k1), "num_instances"): "RdPu"
            for k1 in TABLE_CMAPS_PER_EVALUATION.keys()
        },
    ]

In [18]:
# Determine reference methods for depth 10 tables.
from qaoa_parameter_setting.utils.types import (
    EvaluationType,
    GraphType,
    MethodConfigJSON,
)

reference_methods_10: (
    dict[
        EvaluationType | tuple[Literal["MPS"], bool],
        dict[
            GraphType,
            MethodConfigJSON,
        ],
    ]
    | None
)
if table.problem_class == "MC":
    reference_methods_10 = {
        ("MPS", False): {
            "erdos_renyi": "F_MPS.json",
            "heavy_hex": "F_MPS.json",
            "line_to_full": "LR_MPS_opt.json",
            "random_regular": "FA_MPS_opt.json",
        },
        ("MPS", True): {
            "erdos_renyi": "F_MPSAer.json",
            "heavy_hex": "F_MPSAer.json",
            "line_to_full": "LR_MPSAer_opt.json",
            "random_regular": "FA_MPSAer_opt.json",
        },
        "SV": {
            "erdos_renyi": "TQA_SV_opt.json",
            "heavy_hex": "F_SV.json",
            "line_to_full": "LR_SV_opt.json",
            "random_regular": "TQA_SV_opt.json",
        },
        "PP": {
            "erdos_renyi": "TQA_PP_opt.json",
            "heavy_hex": "TQA_PP_opt.json",
            "line_to_full": "TQA_PP_opt.json",
            "random_regular": "TQA_PP_opt.json",
        },
    }  # type: ignore[assignment]
elif table.problem_class == "MIS":
    reference_methods_10 = None
else:
    raise NotImplementedError(
        "Cannot determine reference methods for problem class {!r}.".format(
            table.problem_class
        )
    )

In [19]:
now = datetime.now(tz=timezone.utc)
generated_on_str = "% Generated on {now:%Y-%m-%d} at {now:%H:%M:%S} UTC\n".format(
    now=now
)
# Open a summary list_of_tables file for previewing all tables.
with open(
    "list_of_tables_{}.tex".format(table.problem_class), "w"
) as f:
    # Write date and time to list_of_tables
    f.write(generated_on_str)

    # Iterate over all depths, sorted so they're included in increasing order in
    # list_of_tables
    for depth in sorted(int(x) for x in table.to_dataframe()["depth"].unique()):
        # Write a section title.
        _ = f.write("\n\\section{{Tables for depth $P={}$}}\n".format(depth))
        for filename_suffix, values, value_label, additional_caption in values_to_plot:
            # This is the filename for this table.
            _latex_filename = "table_{problem_class}_p{depth:02}_{suffix}.tex".format(
                problem_class=table.problem_class,
                depth=int(depth),
                suffix=filename_suffix,
            )
            if depth == 10 and table.problem_class == "MC":
                # Type-narrowing for reference_methods_10
                assert reference_methods_10 is not None
                sub_table = table.only_common_instances(
                    reference_methods_10,
                    per_depth=True,
                )
            else:
                sub_table = table

            if table.problem_class == "MIS":
                sub_table = sub_table.filter_by(
                    num_nodes={
                        "MPS": {True: [70, 100, 144], False: [70, 100, 144]},
                        "PP": [70, 100, 144],
                        "SV": [20],
                    }
                )

            # Get the styler
            _, styler, _ = formatted_styler_for(
                sub_table,
                depths=depth,
                # Select the number of nodes for MIS, only.
                agg_values=values,
                with_fancy_values=True,
                cmap=(  # pyright: ignore[reportArgumentType]
                    table_cmap_num_instances
                    if values == "num_instances"
                    else table_cmap_performance
                )
                if isinstance(values, str)
                else table_cmap_combined,
                precision={performance_metric: 1, "num_instances": 0},
                missing_data_str="-",
                target_format="latex",
                show_empty_rows=True,
            )

            # Save table to separate LaTeX file
            with open(_latex_filename, "w") as f_table:
                _ = f_table.write(generated_on_str)
                _ = f_table.write(
                    "% Data is {value_label} for depth P={depth}.\n".format(
                        value_label=value_label, depth=depth
                    )
                )
                f_table.write(
                    convert_evaluation_to_multicolumn_latex(
                        styler.to_latex(
                            # f_table,
                            convert_css=True,
                            hrules=True,
                            clines="skip-last;data",
                        ),
                        num_metrics=len(values) if isinstance(values, list) else 1,
                    )
                )
            _ = f.write(
                r"""
\begin{{table}}[H]
    \centering
    \input{{{filename}}}
    \caption{{\textbf{{{value_label} for $P={depth}$.}} Graph types are Erdos Renyi (ER), Heavy-Hex (HH), Line-to-Full (LB), and Random Regular (RR).{additional_caption}}}
\end{{table}}
""".format(
                    filename=_latex_filename,
                    depth=depth,
                    value_label=value_label,
                    additional_caption=additional_caption,
                )
            )
            _ = f.write(r"\clearpage")
        # _ = f.write(r"\clearpage")

## Compiling and previewing all tables

Tables are written to `table_p<depth>_<metric>.tex` where `<depth>` is the QAOA
depth and `<metric>` is either `approximation_ratio` or `num_instances`. These
LaTeX files contain the `tabular` environments to include the given tables.
Numerical values are formatted with siunitx for easier precision and uncertainty
handling. All of these tables are then included in `tables.tex` as a list of
sections, one per depth. `summary_tables.tex` is an example LaTeX document that
(i) has an appropriate preamble for rendering the tables and (ii) shows how to
include them in a larger document.

To compile the _sample document_ `summary_tables.tex`, run the following command
after generating the LaTeX files:

```bash
latexmk -pdf summary_tables.tex
```


In [20]:
!latexmk -pdf summary_tables.tex

Rc files read (in order):
  NONE
Latexmk: This is Latexmk, John Collins, 9 March 2026. Version 4.88.
Latexmk: applying rule 'pdflatex'...
Rule 'pdflatex':  Reasons for rerun
Changed files or newly in use/created:
  list_of_tables_MIS.tex
  table_MIS_p01_both.tex
  table_MIS_p01_energy.tex
  table_MIS_p01_num_instances.tex
  table_MIS_p02_both.tex
  table_MIS_p02_energy.tex
  table_MIS_p02_num_instances.tex
  table_MIS_p03_both.tex
  table_MIS_p03_energy.tex
  table_MIS_p03_num_instances.tex
  table_MIS_p04_both.tex
  table_MIS_p04_energy.tex
  table_MIS_p04_num_instances.tex
  table_MIS_p05_both.tex
  table_MIS_p05_energy.tex
  table_MIS_p05_num_instances.tex
  table_MIS_p06_both.tex
  table_MIS_p06_energy.tex
  table_MIS_p06_num_instances.tex
  table_MIS_p07_both.tex
  table_MIS_p07_energy.tex
  table_MIS_p07_num_instances.tex
  table_MIS_p08_both.tex
  table_MIS_p08_energy.tex
  table_MIS_p08_num_instances.tex
  table_MIS_p09_both.tex
  table_MIS_p09_energy.tex
  table_MIS_p09_num_in